**Media SARIMAX at 1, 3 and 6 month horizons**

In [8]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_absolute_error
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.stattools import adfuller
from sklearn.model_selection import train_test_split
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import r2_score
from sklearn.preprocessing import MinMaxScaler
from pmdarima import auto_arima
from sklearn.preprocessing import StandardScaler

from pathlib import Path
import sys
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.fucntions import validate_agg_data, adf_test, Evaluator,rmse

In [3]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def Evaluator(actual, prediction):
    # Convert to Series so we can align/drop NaNs
    actual = pd.Series(actual).reset_index(drop=True)
    prediction = pd.Series(prediction).reset_index(drop=True)

    # Combine into one dataframe
    results = pd.DataFrame({
        "actual": actual,
        "prediction": prediction
    })

    # Remove NaN rows
    results = results.dropna()

    # Calculate metrics
    mae = mean_absolute_error(
        results["actual"],
        results["prediction"]
    )

    rmse = np.sqrt(
        mean_squared_error(
            results["actual"],
            results["prediction"]
        )
    )

    r2 = r2_score(
        results["actual"],
        results["prediction"]
    )

    print(f"Observations used: {len(results)}")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²: {r2:.4f}")

def abline(slope, intercept, color):
    color='maroon'
    axes = plt.gca()
    x_vals = np.array(axes.get_xlim())
    y_vals = intercept + slope * x_vals
    plt.plot(x_vals, y_vals, '--', color=color)

In [11]:
California=pd.read_csv('../data/interim/Media_with_features.csv')
California.tail()

print(California[['year_month']].dtypes)

California['year_month'] = pd.to_datetime(California['year_month']).dt.to_period('M').dt.to_timestamp()

California[['year_month']].dtypes

year_month    str
dtype: object


year_month    datetime64[us]
dtype: object

In [12]:
def adf_test(series, name):

    result = adfuller(series)

    print("="*60)
    print(name)
    print(f"ADF Statistic : {result[0]:.4f}")
    print(f"P-value       : {result[1]:.4f}")

    if result[1] < 0.05:    
        print("Stationary")
    else:
        print("Non-stationary")

cols =['article_count', 'rate_per_100k', 'mean_title_sentiment',
       'mean_title_risk', 'mean_body_risk', 'avg_sentiment', 'avg_risk',
       'mean_body_sentiment', 'tweet_count', 'avg_likes']

for col in cols:
    adf_test(California[col], col)

adf_test(California['rate_per_100k'], "VF Rate per 100k")

article_count
ADF Statistic : -7.0902
P-value       : 0.0000
Stationary
rate_per_100k
ADF Statistic : -0.2003
P-value       : 0.9385
Non-stationary
mean_title_sentiment
ADF Statistic : -11.0878
P-value       : 0.0000
Stationary
mean_title_risk
ADF Statistic : -12.3249
P-value       : 0.0000
Stationary
mean_body_risk
ADF Statistic : -10.8205
P-value       : 0.0000
Stationary
avg_sentiment
ADF Statistic : -4.3870
P-value       : 0.0003
Stationary
avg_risk
ADF Statistic : -3.4830
P-value       : 0.0084
Stationary
mean_body_sentiment
ADF Statistic : -13.7563
P-value       : 0.0000
Stationary
tweet_count
ADF Statistic : -2.3188
P-value       : 0.1660
Non-stationary
avg_likes
ADF Statistic : -2.4387
P-value       : 0.1311
Non-stationary
VF Rate per 100k
ADF Statistic : -0.2003
P-value       : 0.9385
Non-stationary


In [13]:
California['log_rate'] = np.log1p(
    California['rate_per_100k']
)
adf_test(California['log_rate'], "VF Rate per 100k")

California['logdiff_rate']= California['log_rate'].diff()
California= California.dropna(subset= 'logdiff_rate')
adf_test(California['logdiff_rate'], 'VF logdiffed rates')

VF Rate per 100k
ADF Statistic : -0.3003
P-value       : 0.9255
Non-stationary
VF logdiffed rates
ADF Statistic : -8.1790
P-value       : 0.0000
Stationary


In [14]:
#total is 166 months
onem= 165
threem= 163
sixm= 160
horizons=[1,3,6]

In [15]:
X = California.drop(columns=["year_month", "rate_per_100k", "logdiff_rate",'log_rate'])
y = California["logdiff_rate"]

horizons = [1, 3, 6]

splits = {}

for horizon in horizons:
    splits[horizon] = {
        "X_train": X.iloc[:-horizon],
        "X_test": X.iloc[-horizon:],
        "y_train": y.iloc[:-horizon],
        "y_test": y.iloc[-horizon:]
    }

In [16]:
X_train1=splits[1]['X_train']
X_train3=splits[3]['X_train']
X_train6=splits[6]['X_train']
Y_train1=splits[1]['y_train']
Y_train3=splits[3]['y_train']
Y_train6=splits[6]['y_train']
X_test1=splits[1]['X_test']
X_test3=splits[3]['X_test']
X_test6=splits[6]['X_test']
Y_test1=splits[1]['y_test']
Y_test3=splits[3]['y_test']
Y_test6=splits[6]['y_test']

In [16]:
testcols= ['article_count','mean_title_sentiment','mean_title_risk','mean_body_risk','avg_sentiment','avg_risk','mean_body_sentiment', 'tweet_count','avg_likes']
artcols= ['article_count','mean_title_sentiment','mean_title_risk','mean_body_risk']
twcols= ['avg_sentiment','avg_risk','mean_body_sentiment','tweet_count','avg_likes']

In [18]:
data_path = project_root / "data" / "interim" / "Media_with_features.csv"
output_dir = project_root / "results" / "sarimax"

validate_agg_data(
    data=California,
    required_columns=[
        "year_month",
        "rate_per_100k",
        "logdiff_rate",
        *testcols,
    ],
    date_col="year_month"
)

output_dir.mkdir(parents=True, exist_ok=True)

print(f"Loaded {len(California)} rows of data from {data_path}.")

Required-column check passed.
Duplicate-row check passed.
Monthly-continuity check passed.
Loaded 166 rows of data from c:\Users\zions\Cocci\ValleyFever-MediaSignals\data\interim\Media_with_features.csv.


In [13]:
def select_sarimax_order(
        y_train,
        X_train,
        seasonal_period=12,
        trace=False,
):
    automatic_model = auto_arima(
        y=y_train,
        X=X_train,
        seasonal=True,
        m=seasonal_period,
        test="kpss",  # Use KPSS test to determine order of differencing
        seasonal_test="ocsb",  # Use OCSB test to determine seasonal order of differencing
        start_p=0,
        start_q=0,
        max_p=4,
        max_q=4,
        start_P=0,
        start_Q=0,
        max_P=2,
        max_Q=2,
        max_d=2,
        max_D=1,
        stepwise=True,
        information_criterion="aic",
        trace=trace,
    )

    trend = ("c" if automatic_model.with_intercept else "n")

    return {
        "automatic_model": automatic_model,
        "order": automatic_model.order,
        "seasonal_order": automatic_model.seasonal_order,
        "trend": trend
    }

In [20]:
select_sarimax_order(Y_train1, X_train1)

KeyboardInterrupt: 

In [ ]:
# 1-month: train on first 155, test on last 1
One_month_Media = SARIMAX(
    Y_train1,
    exog=X_train1,
    order=(4,0,0),
    seasonal_order=(1,0,0,12),
    trend='n'
).fit()

# Forecast 1 step ahead
forecast = One_month_Media.get_forecast(steps=len(Y_test1), exog=X_test1)
forecast_values = forecast.predicted_mean

c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [ ]:
print(One_month_Media.summary())

                                     SARIMAX Results                                      
Dep. Variable:                       logdiff_rate   No. Observations:                  165
Model:             SARIMAX(4, 0, 0)x(1, 0, 0, 12)   Log Likelihood                 171.398
Date:                            Wed, 07 Oct 2026   AIC                           -312.797
Time:                                    18:11:49   BIC                           -266.208
Sample:                                         0   HQIC                          -293.885
                                            - 165                                         
Covariance Type:                              opg                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
article_count            0.0044      0.003      1.393      0.164      -0.002       0.011
mean_

In [ ]:
def sarimax(X_train, Y_train, X_test, Y_test):
    selected = select_sarimax_order(Y_train, X_train)
    order = selected["order"]
    seasonal_order = selected["seasonal_order"]
    trend = selected["trend"]
    model = SARIMAX(
        endog=Y_train,
        exog=X_train,
        order=order,
        seasonal_order=seasonal_order,
        trend=trend
    ).fit()

    forecast = model.get_forecast(steps=len(Y_test), exog=X_test)
    forecast_values = forecast.predicted_mean    
    evaluation = Evaluator(Y_test, forecast_values)
    return {
    "model": model,
    "forecast": forecast_values,
    "evaluation": evaluation,
    "order": order,
    "seasonal_order": seasonal_order,
    "trend": trend
    }

In [17]:
feature_sets = {
    "all_media": testcols,
    "articles": artcols,
    "twitter": twcols
}

In [ ]:
sarimax_data = {}

horizon_data = {
    "1_month": (X_train1, X_test1, Y_train1, Y_test1),
    "3_month": (X_train3, X_test3, Y_train3, Y_test3),
    "6_month": (X_train6, X_test6, Y_train6, Y_test6)
}

for feature_name, feature_cols in feature_sets.items():

    for horizon, (X_train, X_test, y_train, y_test) in horizon_data.items():

        sarimax_data[feature_name, horizon] = {
            "X_train": X_train[feature_cols],
            "X_test": X_test[feature_cols],
            "y_train": y_train,
            "y_test": y_test,
            "features": feature_cols
        }

In [ ]:
sarimax_results = {}

for (feature_name, horizon), data in sarimax_data.items():

    print(f"\nRunning SARIMAX: {feature_name} - {horizon}")

    sarimax_results[feature_name, horizon] = sarimax(
        X_train=data["X_train"],
        Y_train=data["y_train"],
        X_test=data["X_test"],
        Y_test=data["y_test"]
    )


Running SARIMAX: all_media - 1_month


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


Observations used: 1
MAE: 0.0659
RMSE: 0.0659
R²: nan

Running SARIMAX: all_media - 3_month


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Observations used: 3
MAE: 0.0588
RMSE: 0.0624
R²: -0.1515

Running SARIMAX: all_media - 6_month


...

Observations used: 1
MAE: 0.0327
RMSE: 0.0327
R²: nan

Running SARIMAX: twitter - 3_month


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Observations used: 3
MAE: 0.0464
RMSE: 0.0534
R²: 0.1554

Running SARIMAX: twitter - 6_month


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Observations used: 6
MAE: 0.0345
RMSE: 0.0408
R²: 0.0555


**Have to Do county-by-county, not statewide**

In [ ]:
Tresult1=sarimax_results['twitter','1_month']
Tresult1['evaluation']

In [ ]:
result1=sarimax(X_train1,Y_train1,X_test1,Y_test1)

Observations used: 1
MAE: 0.0659
RMSE: 0.0659
R²: nan


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


In [ ]:
result3=sarimax(X_train3, Y_train3,X_test3,Y_test3)
result6= sarimax(X_train6,Y_train6,X_test6,Y_test6)

c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Observations used: 3
MAE: 0.0588
RMSE: 0.0624
R²: -0.1515
Observations used: 6
MAE: 0.0448
RMSE: 0.0495
R²: -0.3916


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [21]:
def walk_forward_splits(X, y, initial_train=120, horizon=1, step=1):
    splits = []

    for train_end in range(
        initial_train,
        len(X) - horizon + 1,
        step
    ):
        test_end = train_end + horizon

        splits.append({
            "X_train": X.iloc[:train_end],
            "y_train": y.iloc[:train_end],
            "X_test": X.iloc[train_end:test_end],
            "y_test": y.iloc[train_end:test_end]
        })

    return splits

splits = {
    horizon: walk_forward_splits(
        X, y, initial_train=120, horizon=horizon, step=1
    )
    for horizon in [1, 3, 6]
}

In [22]:
feature_sets = {
    "all_media": [
        'article_count',
        'mean_title_sentiment',
        'mean_title_risk',
        'mean_body_risk',
        'avg_sentiment',
        'avg_risk',
        'mean_body_sentiment',
        'tweet_count',
        'avg_likes'
    ],
    "articles": [
        'article_count',
        'mean_title_sentiment',
        'mean_title_risk',
        'mean_body_risk'
    ],
    "twitter": [
        'avg_sentiment',
        'avg_risk',
        'mean_body_sentiment',
        'tweet_count',
        'avg_likes'
    ]
}

In [24]:
sarimax_results = {}

for feature_name, columns in feature_sets.items():

    print(f"\nSelecting SARIMAX order: {feature_name}")

    # Step 1: Use the initial 120 months to select the order
    first_split = splits[1][0]

    X_initial = first_split["X_train"][columns]
    y_initial = first_split["y_train"].squeeze()

    selected = select_sarimax_order(
        y_initial,
        X_initial,
        seasonal_period=12,
        trace=False
    )

    order = selected["order"]
    seasonal_order = selected["seasonal_order"]
    trend = selected["trend"]

    print(f"Order: {order}")
    print(f"Seasonal order: {seasonal_order}")
    print(f"Trend: {trend}")

    # Step 2: Evaluate each forecast horizon
    for horizon in [1, 3, 6]:

        print(f"Walk-forward: {feature_name}, {horizon}-month horizon")

        forecasts = []

        for i, split in enumerate(splits[horizon]):

            X_train = split["X_train"][columns]
            X_test = split["X_test"][columns]

            y_train = np.asarray(split["y_train"]).reshape(-1)
            y_test = np.asarray(split["y_test"]).reshape(-1)

            # Step 3: Fit SARIMAX on the current training window
            model = SARIMAX(
                endog=y_train,
                exog=X_train,
                order=order,
                seasonal_order=seasonal_order,
                trend=trend,
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            fitted_model = model.fit(disp=False)

            # Step 4: Forecast the next 1, 3, or 6 months
            forecast = fitted_model.get_forecast(
                steps=len(y_test),
                exog=X_test
            )

            predictions = np.asarray(
                forecast.predicted_mean
            ).ravel()

            # Step 5: Save actuals and predictions
            for actual, predicted in zip(
                np.asarray(y_test).ravel(),
                predictions
            ):
                forecasts.append({
                    "split": i,
                    "actual": actual,
                    "predicted": predicted
                })

        # Store this feature-set/horizon combination
        sarimax_results[(feature_name, horizon)] = pd.DataFrame(
            forecasts
        )


Selecting SARIMAX order: all_media
Order: (2, 0, 0)
Seasonal order: (1, 0, 0, 12)
Trend: n
Walk-forward: all_media, 1-month horizon


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optim

Walk-forward: all_media, 3-month horizon


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optim

Walk-forward: all_media, 6-month horizon


...

c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optim

Walk-forward: twitter, 3-month horizon


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optim

Walk-forward: twitter, 6-month horizon


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optim

In [25]:
sarimax_evaluation = {}

for (feature_name, horizon), df in sarimax_results.items():

    print(f"\n{feature_name} | {horizon}-month horizon")

    sarimax_evaluation[(feature_name, horizon)] = Evaluator(
        df["actual"].to_numpy(),
        df["predicted"].to_numpy()
    )


all_media | 1-month horizon
Observations used: 46
MAE: 0.0768
RMSE: 0.0971
R²: -0.1310

all_media | 3-month horizon
Observations used: 132
MAE: 0.0776
RMSE: 0.0990
R²: -0.1409

all_media | 6-month horizon
Observations used: 246
MAE: 0.0805
RMSE: 0.1018
R²: -0.1731

articles | 1-month horizon
Observations used: 46
MAE: 0.0739
RMSE: 0.0948
R²: -0.0779

articles | 3-month horizon
Observations used: 132
MAE: 0.0748
RMSE: 0.0965
R²: -0.0841

articles | 6-month horizon
Observations used: 246
MAE: 0.0773
RMSE: 0.0989
R²: -0.1066

twitter | 1-month horizon
Observations used: 46
MAE: 0.0770
RMSE: 0.0967
R²: -0.1214

twitter | 3-month horizon
Observations used: 132
MAE: 0.0783
RMSE: 0.0989
R²: -0.1397

twitter | 6-month horizon
Observations used: 246
MAE: 0.0813
RMSE: 0.1017
R²: -0.1719


In [3]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import r2_score
from sklearn.preprocessing import MinMaxScaler
from pmdarima import auto_arima
from sklearn.preprocessing import StandardScaler

from pathlib import Path
import sys
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.fucntions import validate_agg_data, adf_test, Evaluator,rmse

In [4]:
Counties= pd.read_csv('../data/interim/Media_per_county.csv')

In [7]:
Counties.columns

Index(['County', 'year_month', 'num_cases', 'rate_per_100k',
       'mean_title_positive', 'mean_title_neutral', 'mean_title_negative',
       'mean_title_sentiment', 'mean_body_positive', 'mean_body_neutral',
       'mean_body_negative', 'mean_title_risk', 'mean_body_sentiment',
       'mean_body_risk', 'article_count', 'min_sentiment', 'max_sentiment',
       'avg_sentiment', 'avg_positive', 'avg_negative', 'min_risk', 'max_risk',
       'avg_risk', 'tweet_count', 'avg_likes', 'total_likes'],
      dtype='str')

In [5]:
acols=['mean_title_positive', 'mean_title_neutral', 'mean_title_negative',
       'mean_title_sentiment', 'mean_body_positive', 'mean_body_neutral',
       'mean_body_negative', 'mean_title_risk', 'mean_body_sentiment',
       'mean_body_risk', 'article_count']
tcols=['min_sentiment', 'max_sentiment',
       'avg_sentiment', 'avg_positive', 'avg_negative', 'min_risk', 'max_risk',
       'avg_risk', 'tweet_count', 'avg_likes', 'total_likes']
mcols=['mean_title_positive', 'mean_title_neutral', 'mean_title_negative',
       'mean_title_sentiment', 'mean_body_positive', 'mean_body_neutral',
       'mean_body_negative', 'mean_title_risk', 'mean_body_sentiment',
       'mean_body_risk', 'article_count', 'min_sentiment', 'max_sentiment',
       'avg_sentiment', 'avg_positive', 'avg_negative', 'min_risk', 'max_risk',
       'avg_risk', 'tweet_count', 'avg_likes', 'total_likes']

In [6]:
feature_sets= {'all_media':mcols,
               'articles':acols,
               'twitter':tcols}

features=[mcols,acols,tcols]

target='rate_per_100k'

In [7]:
Counties = Counties.sort_values(
    ['County', 'year_month']
).copy()

county_data = {
    county: group.sort_values('year_month').reset_index(drop=True)
    for county, group in Counties.groupby('County')
}

In [8]:
county_model_data = {}

for county, df in county_data.items():
    df = df.sort_values('year_month').copy()

    df['log_rate'] = np.log1p(df[target])
    df['logdiff_rate'] = df['log_rate'].diff()

    df = df.dropna(subset=['logdiff_rate']).reset_index(drop=True)

    county_model_data[county] = {
        'df': df,
        'y': df['logdiff_rate'],
        'dates': df['year_month'],
        'X': {
            name: df[cols]
            for name, cols in feature_sets.items()
        }
    }

In [12]:
adf_results = {}

for county, data in county_model_data.items():
    y = data['y'].dropna()

    if len(y) < 10 or y.nunique() < 2:
        adf_results[county] = {
            'adf_statistic': np.nan,
            'p_value': np.nan,
            'stationary': None,
            'note': 'Insufficient data or variation'
        }
        continue

    result = adfuller(y, autolag='AIC')

    adf_results[county] = {
        'adf_statistic': result[0],
        'p_value': result[1],
        'stationary': result[1] < 0.05,
        'n_observations': len(y)
    }

adf_df = pd.DataFrame.from_dict(adf_results, orient='index')
adf_df.index.name = 'County'

print(adf_df)

                 adf_statistic       p_value  stationary  n_observations
County                                                                  
Alameda              -4.141295  8.270084e-04        True             155
Contra Costa         -7.259131  1.699679e-10        True             155
Fresno               -3.335289  1.337288e-02        True             155
Kern                 -7.672861  1.576696e-11        True             155
Los Angeles          -5.401650  3.360278e-06        True             155
Merced               -6.792896  2.340142e-09        True             155
Monterey             -6.954466  9.507120e-10        True             155
Orange              -11.877187  6.300479e-22        True             155
Riverside            -8.109161  1.239247e-12        True             155
Sacramento          -10.227624  5.127275e-18        True             155
San Bernardino       -7.371040  8.970604e-11        True             155
San Diego           -10.916924  1.061740e-19       

****Everything above this to loading the csv should be able to be copy pasted for other models**

In [10]:
horizons= [1,3,6]

In [9]:
# Make Splits for every county
county_splits = {}

for county, data in county_model_data.items():
    county_splits[county] = {}

    y = data['y']

    for feature_name, X in data['X'].items():
        county_splits[county][feature_name] = {}

        for horizon in horizons:
            county_splits[county][feature_name][horizon] = {
                "X_train": X.iloc[:-horizon],
                "X_test": X.iloc[-horizon:],
                "y_train": y.iloc[:-horizon],
                "y_test": y.iloc[-horizon:]
            }



In [15]:
county_splits['San Luis Obispo']['all_media'][6]

{'X_train':      mean_title_positive  mean_title_neutral  mean_title_negative  \
 0               0.054337            0.920198             0.025465   
 1               0.032024            0.957100             0.010876   
 2               0.011374            0.272729             0.715897   
 3               0.020025            0.687142             0.292833   
 4               0.026239            0.937853             0.035909   
 ..                   ...                 ...                  ...   
 144             0.038575            0.696140             0.265285   
 145             0.040095            0.630758             0.329147   
 146             0.039491            0.831483             0.129026   
 147             0.042709            0.748463             0.208828   
 148             0.116781            0.547671             0.335548   
 
      mean_title_sentiment  mean_body_positive  mean_body_neutral  \
 0                0.028871            0.026638           0.673269   
 1       

In [18]:
sarimax_results = {}
sarimax_orders = {}

for county, feature_sets_for_county in county_splits.items():

    for feature_name, horizons_for_feature in feature_sets_for_county.items():

        print(f"Selecting SARIMAX order: {county} | {feature_name}")

        # Use the 1-month holdout's training data to select the order
        first_split = horizons_for_feature[1]

        X_initial = first_split["X_train"]
        y_initial = np.asarray(first_split["y_train"]).reshape(-1)

        selected = select_sarimax_order(
            y_initial,
            X_initial,
            seasonal_period=12,
            trace=False
        )

        order = selected["order"]
        seasonal_order = selected["seasonal_order"]
        trend = selected["trend"]

        sarimax_orders[(county, feature_name)] = selected

        print(f"Order: {order}")
        print(f"Seasonal order: {seasonal_order}")
        print(f"Trend: {trend}")

        for horizon, split in horizons_for_feature.items():

            X_train = split["X_train"]
            X_test = split["X_test"]
            y_train = np.asarray(split["y_train"]).reshape(-1)
            y_test = np.asarray(split["y_test"]).reshape(-1)

            model = SARIMAX(
                endog=y_train,
                exog=X_train,
                order=order,
                seasonal_order=seasonal_order,
                trend=trend,
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            fitted_model = model.fit(disp=False)

            forecast = fitted_model.get_forecast(
                steps=len(y_test),
                exog=X_test
            )

            predictions = np.asarray(
                forecast.predicted_mean
            ).reshape(-1)

            sarimax_results[(county, feature_name, horizon)] = pd.DataFrame({
                "actual": y_test,
                "predicted": predictions
            })

Selecting SARIMAX order: Alameda | all_media
Order: (0, 0, 1)
Seasonal order: (0, 0, 0, 12)
Trend: n


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Selecting SARIMAX order: Alameda | articles
Order: (1, 0, 2)
Seasonal order: (0, 0, 0, 12)
Trend: n
Selecting SARIMAX order: Alameda | twitter
Order: (0, 0, 1)
Seasonal order: (0, 0, 0, 12)
Trend: n


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Selecting SARIMAX order: Contra Costa | all_media
Order: (0, 0, 1)
Seasonal order: (1, 0, 0, 12)
Trend: n
Selecting SARIMAX order: Contra Costa | articles
Order: (0, 0, 1)
Seasonal order: (1, 0, 0, 12)
Trend: n


...

c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Selecting SARIMAX order: Ventura | articles
Order: (1, 0, 0)
Seasonal order: (1, 0, 0, 12)
Trend: n


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Selecting SARIMAX order: Ventura | twitter
Order: (1, 0, 0)
Seasonal order: (0, 0, 0, 12)
Trend: n


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [19]:
evaluation_rows = []
for (county, feature_name, horizon), df in sarimax_results.items():
    actual = df["actual"].to_numpy()
    predicted = df["predicted"].to_numpy()

    evaluation_rows.append({
        "County": county,
        "Model": "SARIMAX",
        "Feature Set": feature_name,
        "Horizon": horizon,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": (
            r2_score(actual, predicted)
            if len(actual) >= 2 else np.nan
        ),
        "N Predictions": len(actual)
    })

sarimax_evaluation_table = pd.DataFrame(evaluation_rows)

sarimax_evaluation_table = sarimax_evaluation_table.sort_values(
    ["County", "Horizon", "Feature Set"]
).reset_index(drop=True)

print(sarimax_evaluation_table)

      County    Model Feature Set  Horizon       MAE      RMSE        R2  \
0    Alameda  SARIMAX   all_media        1  0.077027  0.077027       NaN   
1    Alameda  SARIMAX    articles        1  0.089950  0.089950       NaN   
2    Alameda  SARIMAX     twitter        1  0.077418  0.077418       NaN   
3    Alameda  SARIMAX   all_media        3  0.156757  0.157119  0.242794   
4    Alameda  SARIMAX    articles        3  0.163483  0.166132  0.153423   
..       ...      ...         ...      ...       ...       ...       ...   
184  Ventura  SARIMAX    articles        3  0.181145  0.186652 -0.144929   
185  Ventura  SARIMAX     twitter        3  0.235800  0.250722 -1.065860   
186  Ventura  SARIMAX   all_media        6  0.214990  0.225524 -1.163006   
187  Ventura  SARIMAX    articles        6  0.158625  0.180273 -0.382088   
188  Ventura  SARIMAX     twitter        6  0.181233  0.217220 -1.006653   

     N Predictions  
0                1  
1                1  
2                1  
3  

In [11]:
def walk_forward_splits(X, y, initial_train=120, horizon=1, step=1):
    splits = []

    for train_end in range(
        initial_train, len(X) - horizon + 1, step
    ):
        test_end = train_end + horizon

        splits.append({
            "X_train": X.iloc[:train_end],
            "X_test": X.iloc[train_end:test_end],
            "y_train": y.iloc[:train_end],
            "y_test": y.iloc[train_end:test_end]
        })

    return splits


horizons = [1, 3, 6]
initial_train = 120

county_splits = {}

for county, data in county_model_data.items():
    county_splits[county] = {}

    y = data['y']

    for feature_name, X in data['X'].items():
        county_splits[county][feature_name] = {}

        for horizon in horizons:
            county_splits[county][feature_name][horizon] = (
                walk_forward_splits(
                    X,
                    y,
                    initial_train=initial_train,
                    horizon=horizon,
                    step=1
                )
            )

In [ ]:
sarimax_results = {}
sarimax_best_params={}
sarimax_parameter_rows=[]

for county, feature_sets_for_county in county_splits.items():
    for feature_name, horizons_for_feature in feature_sets_for_county.items():
        for horizon, splits_for_horizon in horizons_for_feature.items():

            split_results = []

            for i, split in enumerate(splits_for_horizon):
                X_train = split["X_train"]
                X_test = split["X_test"]
                y_train = split["y_train"]
                y_test = split["y_test"]

                model = SARIMAX(
                    endog=y_train,
                    exog=X_train,
                    order=order,
                    seasonal_order=seasonal_order,
                    trend=trend,
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )

                fitted_model = model.fit(disp=False)

                for feature in X_train.columns:
                    if feature in fitted_model.params.index:
                        sarimax_parameter_rows.append({
                            "County": county,
                            "Feature_Set": feature_name,
                            "Horizon": horizon,
                            "Split": i,
                            "Feature": feature,
                            "Coefficient": fitted_model.params[feature],
                            "P_Value": fitted_model.pvalues[feature]
                        })

                forecast = fitted_model.get_forecast(
                    steps=len(y_test),
                    exog=X_test
                )

                predictions = np.asarray(
                    forecast.predicted_mean
                ).reshape(-1)

                # Save this split instead of overwriting earlier splits
                split_results.append(pd.DataFrame({
                    "split": i + 1,
                    "actual": y_test.to_numpy(),
                    "predicted": predictions
                }))

                print(
                    f"{county} | {feature_name} | horizon={horizon} | "
                    f"{len(splits_for_horizon)} splits complete"
                )

            # Combine ALL walk-forward splits for this combination
            sarimax_results[(county, feature_name, horizon)] = (
                pd.concat(split_results, ignore_index=True)
            )


Alameda all_media horizon=1 split=1/35
Alameda all_media horizon=1 split=2/35


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Alameda all_media horizon=1 split=3/35
Alameda all_media horizon=1 split=4/35
Alameda all_media horizon=1 split=5/35
Alameda all_media horizon=1 split=6/35
Alameda all_media horizon=1 split=7/35


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Alameda all_media horizon=1 split=8/35
Alameda all_media horizon=1 split=9/35


...

c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Ventura twitter horizon=6 split=25/30
Ventura twitter horizon=6 split=26/30


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Ventura twitter horizon=6 split=27/30
Ventura twitter horizon=6 split=28/30
Ventura twitter horizon=6 split=29/30
Ventura twitter horizon=6 split=30/30


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [32]:
evaluation_rows = []
for (county, feature_name, horizon), df in sarimax_results.items():
    actual = df["actual"].to_numpy()
    predicted = df["predicted"].to_numpy()

    evaluation_rows.append({
        "County": county,
        "Model": "SARIMAX",
        "Feature Set": feature_name,
        "Horizon": horizon,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": (
            r2_score(actual, predicted)
            if len(actual) >= 2 else np.nan
        ),
        "N Predictions": len(actual)
    })

sarimax_evaluation_table = pd.DataFrame(evaluation_rows)

sarimax_evaluation_table = sarimax_evaluation_table.sort_values(
    ["County", "Horizon", "Feature Set"]
).reset_index(drop=True)

print(sarimax_evaluation_table)

      County    Model Feature Set  Horizon       MAE      RMSE        R2  \
0    Alameda  SARIMAX   all_media        1  0.161886  0.205833 -0.027434   
1    Alameda  SARIMAX    articles        1  0.156849  0.194077  0.086577   
2    Alameda  SARIMAX     twitter        1  0.152926  0.199510  0.034726   
3    Alameda  SARIMAX   all_media        3  0.172191  0.216797 -0.118239   
4    Alameda  SARIMAX    articles        3  0.165862  0.207826 -0.027612   
..       ...      ...         ...      ...       ...       ...       ...   
184  Ventura  SARIMAX    articles        3  0.224093  0.260958 -0.116741   
185  Ventura  SARIMAX     twitter        3  0.218233  0.260445 -0.112358   
186  Ventura  SARIMAX   all_media        6  0.233980  0.271804 -0.227253   
187  Ventura  SARIMAX    articles        6  0.223082  0.261631 -0.137101   
188  Ventura  SARIMAX     twitter        6  0.217982  0.259007 -0.114412   

     N Predictions  
0               35  
1               35  
2               35  
3  

In [33]:
sarimax_evaluation_table.to_csv('SARIMAX_Media_Evals.csv', index=False)

In [17]:
sarimax_results = {}
sarimax_best_params = {}
sarimax_parameter_rows = []

# -----------------------------------------
# 1. Select parameters ONCE per county
#    and feature set
# -----------------------------------------
for county, feature_sets_for_county in county_splits.items():
    for feature_name, horizons_for_feature in feature_sets_for_county.items():

        # Use the first split from the 1-month horizon
        tuning_splits = horizons_for_feature[1]

        if not tuning_splits:
            print(f"Skipping tuning: {county} | {feature_name}")
            continue

        tuning_split = tuning_splits[0]

        X_tune = tuning_split["X_train"]
        y_tune = tuning_split["y_train"]

        selected = select_sarimax_order(
            y_tune,
            X_tune,
            seasonal_period=12,
            trace=False
        )

        sarimax_best_params[(county, feature_name)] = {
            "order": selected["order"],
            "seasonal_order": selected["seasonal_order"],
            "trend": selected["trend"]
        }

        print(
            f"Tuned: {county} | {feature_name} | "
            f"order={selected['order']} | "
            f"seasonal={selected['seasonal_order']}"
        )


# -----------------------------------------
# 2. Walk-forward evaluation using the
#    previously selected parameters
# -----------------------------------------
for county, feature_sets_for_county in county_splits.items():
    for feature_name, horizons_for_feature in feature_sets_for_county.items():

        # Skip combinations where tuning produced no parameters
        if (county, feature_name) not in sarimax_best_params:
            continue

        # Reuse the same configuration for every horizon and split
        params = sarimax_best_params[(county, feature_name)]

        order = params["order"]
        seasonal_order = params["seasonal_order"]
        trend = params["trend"]

        for horizon, splits_for_horizon in horizons_for_feature.items():

            split_results = []

            for i, split in enumerate(splits_for_horizon):
                X_train = split["X_train"]
                X_test = split["X_test"]
                y_train = split["y_train"]
                y_test = split["y_test"]

                # Refit with current training data, but do not retune
                model = SARIMAX(
                    endog=y_train,
                    exog=X_train,
                    order=order,
                    seasonal_order=seasonal_order,
                    trend=trend,
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )

                fitted_model = model.fit(disp=False)

                # Save coefficients and p-values
                for feature in X_train.columns:
                    if feature in fitted_model.params.index:
                        sarimax_parameter_rows.append({
                            "County": county,
                            "Feature_Set": feature_name,
                            "Horizon": horizon,
                            "Split": i + 1,
                            "Feature": feature,
                            "Coefficient": fitted_model.params[feature],
                            "P_Value": fitted_model.pvalues[feature]
                        })

                # Forecast the test window
                forecast = fitted_model.get_forecast(
                    steps=len(y_test),
                    exog=X_test
                )

                predictions = np.asarray(
                    forecast.predicted_mean
                ).reshape(-1)

                split_results.append(pd.DataFrame({
                    "split": i + 1,
                    "lead": np.arange(1, len(y_test) + 1),
                    "actual": y_test.to_numpy(),
                    "predicted": predictions
                }))

            if split_results:
                sarimax_results[(county, feature_name, horizon)] = (
                    pd.concat(split_results, ignore_index=True)
                )

            print(
                f"Complete: {county} | {feature_name} | "
                f"horizon={horizon}"
            )

Tuned: Alameda | all_media | order=(0, 0, 1) | seasonal=(0, 0, 0, 12)
Tuned: Alameda | articles | order=(2, 0, 2) | seasonal=(0, 0, 0, 12)
Tuned: Alameda | twitter | order=(0, 0, 1) | seasonal=(0, 0, 0, 12)
Tuned: Contra Costa | all_media | order=(0, 0, 1) | seasonal=(0, 0, 0, 12)
Tuned: Contra Costa | articles | order=(0, 0, 1) | seasonal=(1, 0, 0, 12)
Tuned: Contra Costa | twitter | order=(0, 0, 1) | seasonal=(0, 0, 0, 12)
Tuned: Fresno | all_media | order=(0, 0, 0) | seasonal=(1, 0, 0, 12)
Tuned: Fresno | articles | order=(0, 0, 0) | seasonal=(1, 0, 1, 12)
Tuned: Fresno | twitter | order=(0, 0, 0) | seasonal=(0, 0, 0, 12)
Tuned: Kern | all_media | order=(0, 0, 0) | seasonal=(0, 0, 1, 12)
Tuned: Kern | articles | order=(0, 0, 0) | seasonal=(0, 0, 1, 12)
Tuned: Kern | twitter | order=(0, 0, 0) | seasonal=(1, 0, 0, 12)


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:1901: RuntimeWarning: divide by zero encountered in reciprocal
  return np.roots(self.polynomial_reduced_ar)**-1
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:1908: RuntimeWarning: divide by zero encountered in reciprocal
  return np.roots(self.polynomial_reduced_ma)**-1
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:1901: RuntimeWarning: divide by zero encountered in reciprocal
  return np.roots(self.polynomial_reduced_ar)**-1
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:1908: RuntimeWarning: divide by zero encountered in reciprocal
  return np.roots(self.polynomial_reduced_ma)**-1


Tuned: Los Angeles | all_media | order=(1, 0, 1) | seasonal=(0, 0, 0, 12)
Tuned: Los Angeles | articles | order=(0, 0, 3) | seasonal=(0, 0, 0, 12)
Tuned: Los Angeles | twitter | order=(0, 0, 1) | seasonal=(0, 0, 0, 12)
Tuned: Merced | all_media | order=(0, 0, 3) | seasonal=(1, 0, 0, 12)
Tuned: Merced | articles | order=(0, 0, 1) | seasonal=(1, 0, 0, 12)
Tuned: Merced | twitter | order=(0, 0, 1) | seasonal=(2, 0, 0, 12)
Tuned: Monterey | all_media | order=(0, 0, 0) | seasonal=(0, 0, 0, 12)
Tuned: Monterey | articles | order=(1, 0, 1) | seasonal=(1, 0, 0, 12)
Tuned: Monterey | twitter | order=(0, 0, 0) | seasonal=(0, 0, 0, 12)
Tuned: Orange | all_media | order=(1, 0, 0) | seasonal=(1, 0, 0, 12)
Tuned: Orange | articles | order=(0, 0, 1) | seasonal=(0, 0, 2, 12)
Tuned: Orange | twitter | order=(1, 0, 0) | seasonal=(1, 0, 0, 12)
Tuned: Riverside | all_media | order=(1, 0, 0) | seasonal=(0, 0, 0, 12)
Tuned: Riverside | articles | order=(1, 0, 0) | seasonal=(0, 0, 0, 12)
Tuned: Riverside | t

c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:1901: RuntimeWarning: divide by zero encountered in reciprocal
  return np.roots(self.polynomial_reduced_ar)**-1
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:1908: RuntimeWarning: invalid value encountered in reciprocal
  return np.roots(self.polynomial_reduced_ma)**-1
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\pmdarima\arima\_auto_solvers.py:40: RuntimeWarning: invalid value encountered in divide
  max_invroot = max(0, *np.abs(1 / model.maroots()))


Tuned: San Joaquin | all_media | order=(1, 0, 1) | seasonal=(0, 0, 1, 12)
Tuned: San Joaquin | articles | order=(0, 0, 4) | seasonal=(0, 0, 2, 12)
Tuned: San Joaquin | twitter | order=(0, 0, 1) | seasonal=(0, 0, 2, 12)
Tuned: San Luis Obispo | all_media | order=(0, 0, 1) | seasonal=(0, 0, 1, 12)
Tuned: San Luis Obispo | articles | order=(0, 0, 1) | seasonal=(1, 0, 0, 12)
Tuned: San Luis Obispo | twitter | order=(0, 0, 1) | seasonal=(1, 0, 0, 12)
Tuned: San Mateo | all_media | order=(0, 0, 2) | seasonal=(2, 0, 0, 12)
Tuned: San Mateo | articles | order=(0, 0, 1) | seasonal=(0, 0, 0, 12)
Tuned: San Mateo | twitter | order=(0, 0, 1) | seasonal=(0, 0, 0, 12)
Tuned: Santa Barbara | all_media | order=(0, 0, 2) | seasonal=(0, 0, 1, 12)
Tuned: Santa Barbara | articles | order=(2, 0, 2) | seasonal=(0, 0, 1, 12)
Tuned: Santa Barbara | twitter | order=(0, 0, 2) | seasonal=(1, 0, 0, 12)
Tuned: Santa Clara | all_media | order=(0, 0, 1) | seasonal=(0, 0, 0, 12)
Tuned: Santa Clara | articles | order=

...

Complete: Ventura | twitter | horizon=1


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optim

Complete: Ventura | twitter | horizon=3


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optim

Complete: Ventura | twitter | horizon=6


In [18]:
sarimax_predictions_df = pd.concat(
    [
        df.assign(
            County=county,
            Feature_Set=feature_name,
            Horizon=horizon
        )
        for (county, feature_name, horizon), df
        in sarimax_results.items()
    ],
    ignore_index=True
)

sarimax_params_df = pd.DataFrame([
    {
        "County": county,
        "Feature_Set": feature_name,
        "order": str(params["order"]),
        "seasonal_order": str(params["seasonal_order"]),
        "trend": params["trend"]
    }
    for (county, feature_name), params in sarimax_best_params.items()
])

sarimax_params_df.to_csv(
    "../data/interim/sarimax_best_params.csv",
    index=False
)

pd.DataFrame(sarimax_parameter_rows).to_csv(
    "../data/interim/sarimax_coefficients.csv",
    index=False
)

In [19]:
evaluation_rows = []

for (county, feature_name, horizon), df in sarimax_results.items():
    actual = df["actual"].to_numpy()
    predicted = df["predicted"].to_numpy()

    evaluation_rows.append({
        "County": county,
        "Model": "SARIMAX",
        "Feature Set": feature_name,
        "Horizon": horizon,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": (
            r2_score(actual, predicted)
            if len(actual) >= 2 else np.nan
        ),
        "N Predictions": len(actual)
    })

sarimax_evaluations_table = pd.DataFrame(evaluation_rows)

sarimax_evaluations_table = sarimax_evaluations_table.sort_values(
    ["County", "Horizon", "Feature Set"]
).reset_index(drop=True)

sarimax_evaluations_table.to_csv('xgboost_Media_Evals.csv', index=False)

***Try to do models with media features from very beginning to end***